# S17 · Parcial NovaMarket — Lote 08

Preparación reproducible del lote 08. Las solicitudes de los minutos 50 y 80 se recibieron y sus tratamientos y entregables se documentan más adelante.

El CSV final conserva unidades reales, no aplica escalado ni codificación. Las columnas categóricas se conservan normalizadas; las variables de S13 se generan con la información disponible y `fuga_cliente` se reserva como objetivo, no como predictor. `id_pedido` e `id_cliente` se conservan solo como llaves de trazabilidad/agrupación; los correos se excluyen y `fecha_llamada_retencion`, posterior al pedido, no se entrega.

## 1. Carga del lote

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'NovaMarket_datos_crudos.csv').exists():
    RAIZ = RAIZ.parent
S17 = RAIZ / 'S17'
sys.path.insert(0, str(S17))
from funciones_novamarket import limpiar_lote, auditar_lote_vs_semestre, integrar_lote

ruta_lote = S17 / 'NovaMarket_lote_noviembre_08.csv'
ruta_referencia = S17 / 'metricas_semestre.json'
lote_crudo = pd.read_csv(ruta_lote, low_memory=False)
with ruta_referencia.open(encoding='utf-8') as archivo:
    referencia = json.load(archivo)

print(f'Archivo: {ruta_lote.name}')
print(f'Filas: {len(lote_crudo):,} | columnas: {lote_crudo.shape[1]}')
display(lote_crudo.head())
display(lote_crudo.isna().sum().sort_values(ascending=False).to_frame('vacios crudos'))

Archivo: NovaMarket_lote_noviembre_08.csv
Filas: 20,300 | columnas: 21


,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,...,monto_compra,edad_cliente,correo_cliente,nivel_satisfaccion,nivel_lealtad,comentario_cliente,peso_pedido_kg,fecha_actualizacion_stock,fecha_llamada_retencion,fuga_cliente
0,N08006763,K0801732,28/10/2026,Tienda,TRANSFERENCIA,CARTAGENA,1300104,Moda,Zapatos deportivos,109000.0,...,109000,57.0,cliente086763@hotmail.com,Alto,Plata,Rápido y sin problemas,1105.7,2024-07-20,NaN,NaN
1,N08013389,K0801203,17/10/2026,marketplace,PayPal,Medellín,500101,Hogar,Set de cuchillos,263600.0,...,263600,58.0,cliente0813389@gmail.com,Bajo,Plata,Llegó incompleto el pedido,11.6,2026-11-06,NaN,NaN
2,N08009122,K0800401,30/10/2026,Marketplace,Transferencia,bogota,1100106,Moda,Bufanda de lana,440200.0,...,$440.200,26.0,cliente089122@outlook.com,Alto,Platino,Llegó incompleto el pedido,13.3,2026-10-13,NaN,NaN
3,N08010331,K0804751,18/10/2026,Tienda,Transferencia,cucuta,5400107,electrónica,Teclado mecánico,28900.0,...,57800,61.0,cliente0810331@yahoo.es,bajo,Plata,Pésima experiencia con el empaque,1048.3,2026-09-20,2026-12-04,NaN
4,N08011547,K0800045,13/11/2027,Web,transferencia,Santa Marta,4700101,Belleza,Secador de pelo,956300.0,...,1912600,32.0,NaN,Bajo,Plata,Llegó incompleto el pedido,8.3,2026-11-02,NaN,NaN


,vacios crudos
fuga_cliente,16246
fecha_llamada_retencion,14552
correo_cliente,3332
comentario_cliente,2889
edad_cliente,2421
nivel_satisfaccion,1471
ciudad_tienda,950
monto_compra,690
precio_unitario,690
id_pedido,0


## 2. Auditoría del lote frente al semestre

Esta primera comparación es sobre el archivo crudo: las categorías se normalizan solo para comparar, así se distinguen variantes de escritura de categorías realmente nuevas. Las diferencias de columnas y tipos se muestran tal como llegan y se vuelven a revisar después del preproceso.

In [2]:
auditoria_cruda = auditar_lote_vs_semestre(
    lote_crudo, referencia=referencia, titulo='archivo crudo'
)
print('\nMedianas de referencia del semestre:')
display(pd.Series(referencia['semestre']['medianas'], name='mediana_semestre').to_frame())
print('\nPorcentajes de vacíos de referencia del semestre:')
display(pd.Series(referencia['semestre']['porcentaje_vacios'], name='porcentaje_semestre').to_frame())


Auditoria archivo crudo frente al semestre (20,300 filas):
Columnas solo en el lote: ['correo_cliente', 'fecha_llamada_retencion']
Columnas solo en el semestre: ['acepta_promociones', 'antiguedad_cliente_dias', 'bodega_asignada', 'bodega_encontrada', 'categoria', 'ciudad_residencia', 'ciudad_tienda_imputado', 'comentario_cliente_imputado', 'correo_cliente_imputado', 'crm_encontrada', 'dias_desde_compra_previa', 'dias_reposicion', 'edad_cliente_imposible', 'es_venta_corporativa', 'fecha_inscripcion_club', 'gasto_previo_cliente', 'monto_compra_capado', 'monto_compra_recalculado', 'monto_por_unidad', 'nivel_satisfaccion_imputado', 'pedidos_previos_cliente', 'peso_pedido_kg_imposible', 'precio_monto_imputado', 'proveedor_principal', 'puntos_fidelidad_actuales', 'rfm_score', 'segmento_crm', 'stock_minimo', 'ticket_promedio_previo_cliente']
Diferencias de tipos (lote, semestre): {'canal_compra': ('str', 'string'), 'categoria_producto': ('str', 'string'), 'ciudad_tienda': ('str', 'string'), 

Diferencias de categorias: {'canal_compra': {'nuevas_en_lote': ['TikTok Shop'], 'no_observadas_en_lote': []}, 'metodo_pago': {'nuevas_en_lote': ['Contraentrega'], 'no_observadas_en_lote': []}}


Diferencias de medianas (lote vs. semestre): {'precio_unitario': {'mediana_lote': 355500.0, 'mediana_semestre': 305600.0, 'diferencia': 49900.0}, 'unidades_vendidas': {'mediana_lote': 1.0, 'mediana_semestre': 2.0, 'diferencia': -1.0}, 'monto_compra': {'mediana_lote': 527300.0, 'mediana_semestre': 476400.0, 'diferencia': 50900.0}, 'edad_cliente': {'mediana_lote': 47.0, 'mediana_semestre': 46.0, 'diferencia': 1.0}, 'peso_pedido_kg': {'mediana_lote': 11.7, 'mediana_semestre': 11.5, 'diferencia': 0.1999999999999993}}
Diferencias de porcentaje de vacios (puntos porcentuales): {'id_pedido': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'id_cliente': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'fecha_pedido': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 3.549, 'diferencia_pp': -3.549}, 'canal_compra': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'metodo_pago': {'porcentaje_lote': 0.0, 'porcentaje_s

,mediana_semestre
precio_unitario,305600.0
unidades_vendidas,2.0
monto_compra,476400.0
edad_cliente,46.0
peso_pedido_kg,11.5



Porcentajes de vacíos de referencia del semestre:


,porcentaje_semestre
id_pedido,0.0000
id_cliente,0.0000
fecha_pedido,3.5490
canal_compra,0.0000
metodo_pago,0.0000
ciudad_tienda,4.6686
codigo_postal,0.0000
categoria_producto,0.0000
producto,0.0000
precio_unitario,0.0000


### Oportunidad y concentración de vacíos

La función de auditoría compara columnas, tipos, categorías, medianas y porcentaje global de vacíos. Faltaban dos lecturas que las seis dimensiones sí exigen.

La primera es oportunidad: el archivo se llama `lote_noviembre`, así que hay que comprobar que las fechas correspondan a ese periodo y que ningún pedido quede fechado después.

La segunda es completitud por grupo: un porcentaje global parecido al del semestre puede esconder que los vacíos se concentran en un solo canal, que es el mecanismo MAR documentado en la Semana 4.


In [3]:
periodo_declarado = pd.Period('2026-11', freq='M')
fechas_crudas = pd.to_datetime(lote_crudo['fecha_pedido'], errors='coerce', format='mixed')
meses_lote = fechas_crudas.dt.to_period('M')

fechas_sin_convertir = int(fechas_crudas.isna().sum())
pedidos_en_periodo = int((meses_lote == periodo_declarado).sum())
pedidos_antes_periodo = int((meses_lote < periodo_declarado).sum())
pedidos_despues_periodo = int((meses_lote > periodo_declarado).sum())
pedidos_anio_siguiente = int((fechas_crudas.dt.year > periodo_declarado.year).sum())

print(f'Periodo declarado por el nombre del archivo: {periodo_declarado}')
print(f'Dentro del periodo: {pedidos_en_periodo:,} de {len(lote_crudo):,} ({pedidos_en_periodo / len(lote_crudo):.1%})')
print(f'Anteriores al periodo: {pedidos_antes_periodo:,}')
print(f'Posteriores al periodo: {pedidos_despues_periodo:,}, de los cuales {pedidos_anio_siguiente:,} caen en {periodo_declarado.year + 1} o despues')
print(f'Fechas que no convierten: {fechas_sin_convertir:,}')
display(meses_lote.value_counts().sort_index().to_frame('pedidos'))

satisfaccion_vacia = lote_crudo['nivel_satisfaccion'].isna()
canal_normalizado = lote_crudo['canal_compra'].astype('string').str.strip().str.title()
satisfaccion_por_canal = pd.DataFrame({
    'pedidos': canal_normalizado.value_counts(),
    'vacios': satisfaccion_vacia.groupby(canal_normalizado).sum(),
})
satisfaccion_por_canal['porcentaje_vacios'] = (
    satisfaccion_por_canal['vacios'] / satisfaccion_por_canal['pedidos'] * 100
)
satisfaccion_por_canal['share_del_total_de_vacios'] = (
    satisfaccion_por_canal['vacios'] / max(int(satisfaccion_vacia.sum()), 1) * 100
)
satisfaccion_por_canal = satisfaccion_por_canal.sort_values('porcentaje_vacios', ascending=False)

satisfaccion_vacios_pct = float(satisfaccion_vacia.mean() * 100)
satisfaccion_peor_canal = str(satisfaccion_por_canal.index[0])
satisfaccion_peor_canal_pct = float(satisfaccion_por_canal['porcentaje_vacios'].iloc[0])
satisfaccion_share_peor_canal = float(satisfaccion_por_canal['share_del_total_de_vacios'].iloc[0])

# La linea base guarda el esquema ya imputado, donde nivel_satisfaccion quedo en 0% de
# vacios. Para comparar manzanas con manzanas se lee el crudo del semestre.
semestre_satisfaccion = pd.read_csv(
    RAIZ / 'NovaMarket_datos_crudos.csv',
    usecols=['nivel_satisfaccion', 'canal_compra'],
    low_memory=False,
)
vacia_semestre = semestre_satisfaccion['nivel_satisfaccion'].isna()
canal_semestre = semestre_satisfaccion['canal_compra'].astype('string').str.strip().str.title()
satisfaccion_semestre_pct = float(vacia_semestre.mean() * 100)
share_semestre_por_canal = (
    vacia_semestre.groupby(canal_semestre).sum() / max(int(vacia_semestre.sum()), 1) * 100
).sort_values(ascending=False)
satisfaccion_peor_canal_semestre = str(share_semestre_por_canal.index[0])
satisfaccion_share_semestre_peor = float(share_semestre_por_canal.iloc[0])

print(f'\nnivel_satisfaccion vacia: {satisfaccion_vacios_pct:.2f}% del lote frente a {satisfaccion_semestre_pct:.2f}% del semestre')
print(f'Concentracion: {satisfaccion_peor_canal} tiene {satisfaccion_peor_canal_pct:.1f}% de vacios y concentra el {satisfaccion_share_peor_canal:.1f}% de todos los vacios del lote.')
print(
    f'En el semestre el canal que mas concentraba vacios era {satisfaccion_peor_canal_semestre} '
    f'con el {satisfaccion_share_semestre_peor:.1f}%: el mecanismo MAR de S04 sigue vigente.'
)
display(satisfaccion_por_canal)


Periodo declarado por el nombre del archivo: 2026-11


Dentro del periodo: 1,736 de 20,300 (8.6%)
Anteriores al periodo: 17,018
Posteriores al periodo: 828, de los cuales 526 caen en 2027 o despues
Fechas que no convierten: 718


,pedidos
fecha_pedido,
2026-01,609
2026-02,662
2026-03,662
2026-04,619
2026-05,662
2026-06,644
2026-07,662
2026-08,608
2026-09,606



nivel_satisfaccion vacia: 7.25% del lote frente a 7.26% del semestre
Concentracion: App tiene 20.7% de vacios y concentra el 65.2% de todos los vacios del lote.
En el semestre el canal que mas concentraba vacios era App con el 69.9%: el mecanismo MAR de S04 sigue vigente.


,pedidos,vacios,porcentaje_vacios,share_del_total_de_vacios
canal_compra,,,,
App,4630,959,20.712743,65.193746
Tiktok Shop,1555,109,7.009646,7.409925
Tienda,4806,152,3.162713,10.333107
Web,4595,134,2.916213,9.109449
Marketplace,4714,117,2.481969,7.953773


## 3. Preproceso

`limpiar_lote` elimina duplicados exactos, convierte y reporta conversiones que crean vacíos, normaliza texto, trata valores imposibles y faltantes, integra CRM/bodega con validación `m:1` y crea las variables de S13. No se crean columnas codificadas de S09 porque el entregable exige categorías legibles y unidades originales.

In [4]:
lote_limpio = limpiar_lote(lote_crudo)
filas_eliminadas_por_imposibles = int(lote_limpio.attrs.get('filas_eliminadas_imposibles', 0))

# El CRM del semestre (S12) se une por correo y no empareja ningun pedido de este lote:
# sus columnas quedarian 100% vacias en el entregable, asi que no se entregan.
# antiguedad_cliente_dias depende de fecha_inscripcion_club y se recalcula en la seccion 5.
columnas_crm_semestre = [
    'ciudad_residencia', 'fecha_inscripcion_club', 'puntos_fidelidad_actuales',
    'segmento_crm', 'acepta_promociones', 'crm_encontrada', 'antiguedad_cliente_dias',
]
if not bool(lote_limpio['crm_encontrada'].any()):
    presentes = [c for c in columnas_crm_semestre if c in lote_limpio.columns]
    print(f'CRM del semestre: 0 pedidos emparejados; se retiran {len(presentes)} columnas sin informacion.')
    print(f'  {presentes}')
    lote_limpio = lote_limpio.drop(columns=presentes)

ruta_salida = S17 / 'S17_Sudo_Sancocho_Lote08_preprocesado.csv'
lote_limpio.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print(f'Entregable escrito: {ruta_salida.name} ({len(lote_limpio):,} filas, {lote_limpio.shape[1]} columnas)')


Lote crudo: 20,300 filas, 21 columnas.
Duplicados exactos: 300; filas tras eliminarlos: 20,000.
Conversion id_pedido: 0 valores no vacios pasaron a vacios.
Conversion id_cliente: 0 valores no vacios pasaron a vacios.
Conversion codigo_postal: 0 valores no vacios pasaron a vacios.
Conversion precio_unitario: 0 valores no vacios pasaron a vacios.
Conversion unidades_vendidas: 0 valores no vacios pasaron a vacios.
Conversion edad_cliente: 0 valores no vacios pasaron a vacios.


Conversion peso_pedido_kg: 0 valores no vacios pasaron a vacios.
Conversion monto_compra: 0 valores no vacios pasaron a vacios.
Conversion fecha_pedido: 710 valores no vacios pasaron a vacios.
Conversion fecha_actualizacion_stock: 0 valores no vacios pasaron a vacios.


Conversion fecha_llamada_retencion: 0 valores no vacios pasaron a vacios.
Conversion fuga_cliente: 0 valores no vacios pasaron a vacios.


Normalizacion canal_compra: 0 vacios nuevos; 5 categorias.


Normalizacion metodo_pago: 0 vacios nuevos; 5 categorias.


Normalizacion ciudad_tienda: 0 vacios nuevos; 12 categorias.


Normalizacion categoria_producto: 0 vacios nuevos; 6 categorias.
Normalizacion nivel_satisfaccion: 0 vacios nuevos; 3 categorias.


Normalizacion nivel_lealtad: 0 vacios nuevos; 4 categorias.
Valores imposibles de pedido: precio negativo/centinela=807, unidades no positivas/centinela=1,452, monto negativo=1,160; filas eliminadas (interseccion incluida)=2,196.
Regla monto_compra = precio_unitario x unidades_vendidas: 0 valores discrepantes; se recalculan.
Edad fuera de 0-100: 659; peso mayor de 100 kg: 900; se conservan los pedidos y se dejan esos valores vacios.
Precio imputado con mediana semestral (305,600.00): 636; monto recalculado/imputado: 636.
Satisfaccion imputada por moda semestral del canal: 1,293 (incluye canal nuevo con moda global si aplica).
Correo completado con marcador explicito: 2,907; comentarios completados: 2,504; ciudad sin imputar (con indicador): 819.
Outliers segun IQR semestral: corporativas marcadas=995; montos capados con limite 1,669,600.00=844.


Integracion crm: filas antes=17,804, despues=17,804; llaves sin pareja=17,804; llaves repetidas en tabla=0.
Conversion fecha_inscripcion_club: 0 valores no vacios pasaron a vacios.
Integracion bodega: filas antes=17,804, despues=17,804; llaves sin pareja=2,946; llaves repetidas en tabla=0.
Features de historial calculadas solo dentro del lote: 17,169 filas con fecha/cliente.
Columna excluida por ser posterior al pedido: fecha_llamada_retencion.
Columnas personales usadas solo para CRM y excluidas de la salida: correo_cliente, email_cliente.
Limpieza finalizada: 17,804 filas (inicio 20,300), 48 columnas; pedidos unicos=17,804.
CRM del semestre: 0 pedidos emparejados; se retiran 7 columnas sin informacion.
  ['ciudad_residencia', 'fecha_inscripcion_club', 'puntos_fidelidad_actuales', 'segmento_crm', 'acepta_promociones', 'crm_encontrada', 'antiguedad_cliente_dias']


Entregable escrito: S17_Sudo_Sancocho_Lote08_preprocesado.csv (17,804 filas, 41 columnas)


## 4. Verificación del resultado

Se verifica el lote real y, además, una muestra reproducible de 5.000 filas del CSV crudo del semestre procesada por la misma función.

In [5]:
def verificar_resultado(df, nombre, estricto=True, vacias_esperadas=()):
    print(f'\n{nombre}: {len(df):,} filas, {df.shape[1]} columnas')
    print('Duplicados exactos:', int(df.duplicated().sum()))
    print('id_pedido repetidos:', int(df['id_pedido'].duplicated().sum()))
    print('Tipos:')
    display(df.dtypes.astype(str).to_frame('dtype'))
    print('Vacíos:')
    display(df.isna().sum().sort_values(ascending=False).to_frame('cantidad'))
    for columna in ['canal_compra', 'metodo_pago', 'categoria_producto', 'ciudad_tienda', 'nivel_satisfaccion', 'nivel_lealtad']:
        if columna in df.columns:
            print(f'{columna}: {sorted(df[columna].dropna().astype(str).unique())}')
    vacias_totales = [columna for columna in df.columns if df[columna].isna().all()]
    print('Columnas 100% vacias:', vacias_totales or 'ninguna')
    if estricto:
        inesperadas = [c for c in vacias_totales if c not in vacias_esperadas]
        assert not inesperadas, f'{nombre} entrega columnas 100% vacias: {inesperadas}'
    assert df['id_pedido'].is_unique
    assert df.duplicated().sum() == 0
    assert pd.api.types.is_datetime64_any_dtype(df['fecha_pedido'])
    assert pd.api.types.is_integer_dtype(df['unidades_vendidas'])
    assert pd.api.types.is_numeric_dtype(df['precio_unitario'])
    assert pd.api.types.is_numeric_dtype(df['monto_compra'])
    assert df['unidades_vendidas'].gt(0).all()

verificar_resultado(lote_limpio, 'Lote noviembre limpio')
auditoria_limpia = auditar_lote_vs_semestre(
    lote_limpio, referencia=referencia, titulo='lote limpio'
)

crudo_semestre = pd.read_csv(RAIZ / 'NovaMarket_datos_crudos.csv', low_memory=False)
clientes_lote = set(lote_limpio['id_cliente'].dropna().astype(str))
clientes_semestre = set(crudo_semestre['id_cliente'].dropna().astype(str))
clientes_compartidos = len(clientes_lote & clientes_semestre)
print(f'Clientes con historial en el semestre: {clientes_compartidos:,} de {len(clientes_lote):,} del lote.')
muestra_cruda = crudo_semestre.sample(n=5000, random_state=42).copy()
print(f'\nMuestra cruda del semestre: {len(muestra_cruda):,} filas; semilla=42')
muestra_limpia = limpiar_lote(muestra_cruda)
verificar_resultado(muestra_limpia, 'Muestra del semestre limpia', estricto=False)
auditoria_muestra = auditar_lote_vs_semestre(
    muestra_limpia, referencia=referencia, titulo='muestra de 5.000 filas limpia'
)
print('Columnas nuevas o ausentes frente al esquema de referencia:')
print(auditoria_limpia['columnas_solo_lote'], auditoria_limpia['columnas_solo_semestre'])


Lote noviembre limpio: 17,804 filas, 41 columnas
Duplicados exactos: 0
id_pedido repetidos: 0
Tipos:


,dtype
id_pedido,string
id_cliente,string
fecha_pedido,datetime64[us]
canal_compra,string
metodo_pago,string
ciudad_tienda,string
codigo_postal,string
categoria_producto,string
producto,string
precio_unitario,float64


Vacíos:


,cantidad
fuga_cliente,14249
ticket_promedio_previo_cliente,5721
dias_desde_compra_previa,5721
proveedor_principal,2946
bodega_asignada,2946
categoria,2946
stock_minimo,2946
dias_reposicion,2946
edad_cliente,2769
peso_pedido_kg,900


canal_compra: ['App', 'Marketplace', 'Tienda', 'TikTok Shop', 'Web']
metodo_pago: ['Contraentrega', 'Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']
categoria_producto: ['Belleza', 'Deportes', 'Electronica', 'Hogar', 'Jugueteria', 'Moda']
ciudad_tienda: ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']
nivel_satisfaccion: ['Alto', 'Bajo', 'Medio']
nivel_lealtad: ['Bronce', 'Oro', 'Plata', 'Platino']
Columnas 100% vacias: ninguna

Auditoria lote limpio frente al semestre (17,804 filas):
Columnas solo en el lote: ninguna
Columnas solo en el semestre: ['acepta_promociones', 'antiguedad_cliente_dias', 'ciudad_residencia', 'crm_encontrada', 'fecha_inscripcion_club', 'puntos_fidelidad_actuales', 'segmento_crm']
Diferencias de tipos (lote, semestre): {'bodega_asignada': ('str', 'object'), 'categoria': ('str', 'object'), 'fecha_actualizacion_stock': ('datetime64[us]', 'datetime64[ns]'), 'fecha

Diferencias de categorias: {'canal_compra': {'nuevas_en_lote': ['TikTok Shop'], 'no_observadas_en_lote': []}, 'metodo_pago': {'nuevas_en_lote': ['Contraentrega'], 'no_observadas_en_lote': []}}
Diferencias de medianas (lote vs. semestre): {'precio_unitario': {'mediana_lote': 350450.0, 'mediana_semestre': 305600.0, 'diferencia': 44850.0}, 'unidades_vendidas': {'mediana_lote': 1.0, 'mediana_semestre': 2.0, 'diferencia': -1.0}, 'monto_compra': {'mediana_lote': 550200.0, 'mediana_semestre': 476400.0, 'diferencia': 73800.0}, 'edad_cliente': {'mediana_lote': 46.0, 'mediana_semestre': 46.0, 'diferencia': 0.0}, 'peso_pedido_kg': {'mediana_lote': 11.5, 'mediana_semestre': 11.5, 'diferencia': 0.0}}
Diferencias de porcentaje de vacios (puntos porcentuales): {'id_pedido': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'id_cliente': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'fecha_pedido': {'porcentaje_lote': 3.5666142439901147, 'porcent

... (8 outputs left out)

Vacíos:


,cantidad
ticket_promedio_previo_cliente,4426
dias_desde_compra_previa,4426
antiguedad_cliente_dias,2569
acepta_promociones,1528
puntos_fidelidad_actuales,1528
fecha_inscripcion_club,1528
ciudad_residencia,1528
segmento_crm,1528
bodega_asignada,782
dias_reposicion,782


canal_compra: ['App', 'Marketplace', 'Tienda', 'Web']
metodo_pago: ['Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']
categoria_producto: ['Belleza', 'Deportes', 'Electronica', 'Hogar', 'Jugueteria', 'Moda']
ciudad_tienda: ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']
nivel_satisfaccion: ['Alto', 'Bajo', 'Medio']
nivel_lealtad: ['Bronce', 'Oro', 'Plata', 'Platino']
Columnas 100% vacias: ninguna

Auditoria muestra de 5.000 filas limpia frente al semestre (4,489 filas):
Columnas solo en el lote: ninguna
Columnas solo en el semestre: ninguna
Diferencias de tipos (lote, semestre): {'acepta_promociones': ('str', 'object'), 'bodega_asignada': ('str', 'object'), 'categoria': ('str', 'object'), 'ciudad_residencia': ('str', 'object'), 'fecha_actualizacion_stock': ('datetime64[us]', 'datetime64[ns]'), 'fecha_inscripcion_club': ('datetime64[us]', 'datetime64[ns]'), 'fecha_pedido': ('datetime64

Diferencias de categorias: ninguna
Diferencias de medianas (lote vs. semestre): {'precio_unitario': {'mediana_lote': 305600.0, 'mediana_semestre': 305600.0, 'diferencia': 0.0}, 'unidades_vendidas': {'mediana_lote': 1.0, 'mediana_semestre': 2.0, 'diferencia': -1.0}, 'monto_compra': {'mediana_lote': 477900.0, 'mediana_semestre': 476400.0, 'diferencia': 1500.0}, 'edad_cliente': {'mediana_lote': 47.0, 'mediana_semestre': 46.0, 'diferencia': 1.0}, 'peso_pedido_kg': {'mediana_lote': 11.5, 'mediana_semestre': 11.5, 'diferencia': 0.0}}
Diferencias de porcentaje de vacios (puntos porcentuales): {'id_pedido': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'id_cliente': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'fecha_pedido': {'porcentaje_lote': 3.6088215638226777, 'porcentaje_semestre': 3.549, 'diferencia_pp': 0.05982156382267778}, 'canal_compra': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'metodo_pa

## 5. Solicitudes de Camila

### Minuto 50 — Integración con el CRM
- **Qué pidió:** entregar `S17_Sudo_Sancocho_Lote08_integrado.csv` sin perder ni duplicar pedidos e informar cobertura y registros sin pareja.
- **Qué se entregó:** unión izquierda validada `m:1` con el CRM del lote 08. Se eligió el registro más reciente por `fecha_actualizacion` para las llaves repetidas; `k0802366` tiene empate en esa fecha con valores distintos y se excluyeron sus dos registros, sin escoger uno arbitrariamente.
- **Qué no se pudo resolver:** 5 pedidos de esa llave quedan sin pareja. Se conservaron los 17.804 pedidos; la marca `crm_encontrada` y `crm_actualizacion_posterior_pedido` documentan cobertura y riesgo temporal.
- **Limitación:** 555 pedidos emparejados tienen `fecha_actualizacion` posterior a `fecha_pedido`; los campos del CRM no deben usarse como predictores históricos en esas filas sin una fuente temporal adecuada.

### Tienda nueva — Minuto 80
- **Qué pidió:** preprocesar los pedidos de Popayán con las mismas reglas, columnas y formatos del lote, para poder anexarlos.
- **Qué se cambió en el código y por qué:** `limpiar_lote` normaliza encabezados en mayúsculas y traduce el alias `CANAL` a `canal_compra`; el parser acepta fechas `AAAA.MM.DD` y el peso decimal con coma. Son diferencias del formato de exportación, no cambios de significado.
- **Qué se entregó:** `S17_Sudo_Sancocho_Lote08_tienda_nueva.csv`, homologado al esquema integrado de 49 columnas.
- **Qué no se pudo dejar igual:** el archivo no trae `fuga_cliente`; se agrega la columna vacía, sin inventar etiquetas. El CRM encuentra 420 pedidos y deja 2.580 sin pareja; en 12 de los 420 emparejados la actualización es posterior al pedido. Además, 503 pedidos de Belleza no encuentran categoría en el catálogo de bodega; quedan marcados y sin asignación.

In [6]:
import unicodedata

ruta_crm = S17 / 'NovaMarket_crm_lote_08.csv'
crm_crudo = pd.read_csv(ruta_crm, low_memory=False, dtype={'cliente_id': 'string'})
crm = crm_crudo.copy()
crm['cliente_id'] = crm['cliente_id'].str.strip()
if crm['cliente_id'].isna().any() or crm['cliente_id'].eq('').any():
    raise ValueError('El CRM contiene llaves cliente_id vacías; no se puede validar la unión.')
crm['_llave'] = crm['cliente_id'].str.casefold()

for columna in ['fecha_inscripcion_club', 'fecha_actualizacion']:
    antes = crm[columna].notna()
    crm[columna] = pd.to_datetime(crm[columna], errors='coerce')
    nuevos_vacios = int((antes & crm[columna].isna()).sum())
    print(f'{columna}: {nuevos_vacios:,} valores no vacíos se volvieron NaT')
antes = crm['puntos_fidelidad'].notna()
crm['puntos_fidelidad'] = pd.to_numeric(crm['puntos_fidelidad'], errors='coerce')
nuevos_vacios = int((antes & crm['puntos_fidelidad'].isna()).sum())
print(f'puntos_fidelidad: {nuevos_vacios:,} valores no vacíos se volvieron vacíos')
if crm['fecha_actualizacion'].isna().any():
    raise ValueError('Hay llaves sin fecha_actualizacion; no se puede escoger el registro vigente.')

def normalizar_segmento(valor):
    if pd.isna(valor):
        return pd.NA
    texto = str(valor).strip()
    clave = ''.join(
        caracter for caracter in unicodedata.normalize('NFKD', texto.casefold())
        if not unicodedata.combining(caracter)
    )
    canonicos = {'basico': 'Básico', 'estandar': 'Estándar', 'premium': 'Premium'}
    return canonicos.get(clave, texto.title())

crm['segmento_crm'] = crm['segmento_crm'].map(normalizar_segmento)
llaves_repetidas = crm.loc[crm['_llave'].duplicated(keep=False), '_llave'].nunique()
filas_repetidas = int(crm['_llave'].duplicated(keep=False).sum())
print(f'CRM crudo: {len(crm):,} filas; {llaves_repetidas:,} llaves repetidas en {filas_repetidas:,} filas')

ultima_fecha = crm.groupby('_llave')['fecha_actualizacion'].transform('max')
crm_ultima = crm.loc[crm['fecha_actualizacion'].eq(ultima_fecha)].copy()
llaves_con_empate = crm_ultima.loc[crm_ultima['_llave'].duplicated(keep=False), '_llave'].nunique()
columnas_conflicto = ['segmento_crm', 'puntos_fidelidad', 'fecha_inscripcion_club']
variaciones = crm_ultima.groupby('_llave')[columnas_conflicto].nunique(dropna=False).gt(1).any(axis=1)
llaves_ambiguas = set(variaciones[variaciones].index)
print(f'Llaves empatadas en la última fecha: {llaves_con_empate:,}; con valores conflictivos: {len(llaves_ambiguas):,}')
print(f'Llaves conflictivas excluidas: {sorted(llaves_ambiguas)}')
pedidos_llave_ambigua = int(
    lote_limpio['id_cliente'].astype('string').str.strip().str.casefold().isin(llaves_ambiguas).sum()
)
print(f'Pedidos afectados por llaves ambiguas: {pedidos_llave_ambigua:,}')
crm_resuelto = (
    crm_ultima.loc[~crm_ultima['_llave'].isin(llaves_ambiguas)]
    .drop_duplicates('_llave', keep='first')
    .drop(columns='_llave')
)
assert crm_resuelto['cliente_id'].str.casefold().is_unique
display(crm_resuelto['segmento_crm'].value_counts(dropna=False).to_frame('clientes'))

columnas_crm_antiguo = [
    'ciudad_residencia', 'fecha_inscripcion_club', 'puntos_fidelidad_actuales',
    'segmento_crm', 'acepta_promociones', 'crm_encontrada'
]
columnas_crm_antiguo = [columna for columna in columnas_crm_antiguo if columna in lote_limpio.columns]
columnas_con_datos = [
    columna for columna in columnas_crm_antiguo
    if columna != 'crm_encontrada' and lote_limpio[columna].notna().any()
]
if columnas_con_datos or lote_limpio.get('crm_encontrada', pd.Series(False, index=lote_limpio.index)).fillna(False).any():
    raise ValueError(f'No se pueden sustituir campos CRM anteriores con datos: {columnas_con_datos}')
lote_para_crm = lote_limpio.drop(columns=columnas_crm_antiguo)
lote_integrado = integrar_lote(
    lote_para_crm, crm_resuelto, llave_lote='id_cliente', llave_tabla='cliente_id', nombre='crm'
)
lote_integrado['puntos_fidelidad'] = lote_integrado['puntos_fidelidad'].astype('Int64')
assert len(lote_integrado) == len(lote_limpio)
assert lote_integrado['id_pedido'].is_unique

fechas_comparables = (
    lote_integrado['crm_encontrada']
    & lote_integrado['fecha_pedido'].notna()
    & lote_integrado['fecha_actualizacion'].notna()
)
lote_integrado['crm_actualizacion_posterior_pedido'] = pd.Series(
    pd.NA, index=lote_integrado.index, dtype='boolean'
)
lote_integrado.loc[fechas_comparables, 'crm_actualizacion_posterior_pedido'] = (
    lote_integrado.loc[fechas_comparables, 'fecha_actualizacion']
    > lote_integrado.loc[fechas_comparables, 'fecha_pedido']
)
coincidencias = int(lote_integrado['crm_encontrada'].sum())
sin_pareja_crm = int((~lote_integrado['crm_encontrada']).sum())
actualizaciones_posteriores = int(lote_integrado['crm_actualizacion_posterior_pedido'].fillna(False).sum())
print(f'Pedidos con pareja: {coincidencias:,}; sin pareja: {sin_pareja_crm:,} de {len(lote_integrado):,}')
print(
    f'Actualización CRM posterior al pedido: {actualizaciones_posteriores:,} de '
    f'{int(fechas_comparables.sum()):,} pedidos con ambas fechas observadas.'
)

# Con fecha_inscripcion_club del CRM del lote ya se puede calcular la antiguedad,
# que el preproceso no pudo obtener porque el CRM del semestre no emparejo.
lote_integrado['antiguedad_cliente_dias'] = (
    lote_integrado['fecha_pedido'] - lote_integrado['fecha_inscripcion_club']
).dt.days
lote_integrado['antiguedad_cliente_dias'] = lote_integrado['antiguedad_cliente_dias'].where(
    lote_integrado['antiguedad_cliente_dias'].ge(0)
)
print(
    f"antiguedad_cliente_dias calculada en {int(lote_integrado['antiguedad_cliente_dias'].notna().sum()):,} "
    f'de {len(lote_integrado):,} pedidos.'
)

# cliente_id es la llave del CRM y repite id_cliente en las filas emparejadas;
# crm_encontrada ya documenta la cobertura, asi que no se entrega duplicada.
lote_integrado = lote_integrado.drop(columns='cliente_id')

verificar_resultado(lote_integrado, 'Lote 08 integrado')

ruta_integrado = S17 / 'S17_Sudo_Sancocho_Lote08_integrado.csv'
lote_integrado.to_csv(ruta_integrado, index=False, encoding='utf-8-sig')
print(f'Entregable escrito: {ruta_integrado.name} ({len(lote_integrado):,} filas)')

fecha_inscripcion_club: 0 valores no vacíos se volvieron NaT
fecha_actualizacion: 0 valores no vacíos se volvieron NaT
puntos_fidelidad: 0 valores no vacíos se volvieron vacíos
CRM crudo: 5,232 filas; 509 llaves repetidas en 1,018 filas
Llaves empatadas en la última fecha: 1; con valores conflictivos: 1
Llaves conflictivas excluidas: ['k0802366']
Pedidos afectados por llaves ambiguas: 5


,clientes
segmento_crm,
Premium,1594
Estándar,1564
Básico,1564


Integracion crm: filas antes=17,804, despues=17,804; llaves sin pareja=1,849; llaves repetidas en tabla=0.
Pedidos con pareja: 15,955; sin pareja: 1,849 de 17,804
Actualización CRM posterior al pedido: 555 de 15,400 pedidos con ambas fechas observadas.
antiguedad_cliente_dias calculada en 15,400 de 17,804 pedidos.

Lote 08 integrado: 17,804 filas, 48 columnas
Duplicados exactos: 0
id_pedido repetidos: 0
Tipos:


,dtype
id_pedido,string
id_cliente,string
fecha_pedido,datetime64[us]
canal_compra,string
metodo_pago,string
ciudad_tienda,string
codigo_postal,string
categoria_producto,string
producto,string
precio_unitario,float64


Vacíos:


,cantidad
fuga_cliente,14249
ticket_promedio_previo_cliente,5721
dias_desde_compra_previa,5721
proveedor_principal,2946
categoria,2946
bodega_asignada,2946
dias_reposicion,2946
stock_minimo,2946
edad_cliente,2769
crm_actualizacion_posterior_pedido,2404


canal_compra: ['App', 'Marketplace', 'Tienda', 'TikTok Shop', 'Web']
metodo_pago: ['Contraentrega', 'Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']
categoria_producto: ['Belleza', 'Deportes', 'Electronica', 'Hogar', 'Jugueteria', 'Moda']
ciudad_tienda: ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']
nivel_satisfaccion: ['Alto', 'Bajo', 'Medio']
nivel_lealtad: ['Bronce', 'Oro', 'Plata', 'Platino']
Columnas 100% vacias: ninguna


Entregable escrito: S17_Sudo_Sancocho_Lote08_integrado.csv (17,804 filas)


In [7]:
ruta_tienda_cruda = S17 / 'NovaMarket_tienda_nueva_08.csv'
tienda_cruda = pd.read_csv(ruta_tienda_cruda, low_memory=False)
print(f'Archivo de tienda nueva: {len(tienda_cruda):,} filas y {tienda_cruda.shape[1]} columnas')
tienda_sin_etiqueta_fuente = 'fuga_cliente' not in tienda_cruda.columns
tienda_limpia = limpiar_lote(tienda_cruda)
if 'fuga_cliente' not in tienda_limpia.columns:
    tienda_limpia['fuga_cliente'] = pd.Series(pd.NA, index=tienda_limpia.index, dtype='Int64')

columnas_crm_anterior_tienda = [
    'ciudad_residencia', 'fecha_inscripcion_club', 'puntos_fidelidad_actuales',
    'segmento_crm', 'acepta_promociones', 'crm_encontrada'
]
columnas_crm_anterior_tienda = [
    columna for columna in columnas_crm_anterior_tienda if columna in tienda_limpia.columns
]
columnas_crm_tienda_con_datos = [
    columna for columna in columnas_crm_anterior_tienda
    if columna != 'crm_encontrada' and tienda_limpia[columna].notna().any()
]
if columnas_crm_tienda_con_datos or tienda_limpia.get(
    'crm_encontrada', pd.Series(False, index=tienda_limpia.index)
).fillna(False).any():
    raise ValueError(f'No se pueden sustituir campos CRM anteriores con datos: {columnas_crm_tienda_con_datos}')
tienda_para_crm = tienda_limpia.drop(columns=columnas_crm_anterior_tienda)
tienda_integrada = integrar_lote(
    tienda_para_crm, crm_resuelto, llave_lote='id_cliente', llave_tabla='cliente_id', nombre='crm'
)
tienda_integrada['puntos_fidelidad'] = tienda_integrada['puntos_fidelidad'].astype('Int64')
fechas_comparables_tienda = (
    tienda_integrada['crm_encontrada']
    & tienda_integrada['fecha_pedido'].notna()
    & tienda_integrada['fecha_actualizacion'].notna()
)
tienda_integrada['crm_actualizacion_posterior_pedido'] = pd.Series(
    pd.NA, index=tienda_integrada.index, dtype='boolean'
)
tienda_integrada.loc[fechas_comparables_tienda, 'crm_actualizacion_posterior_pedido'] = (
    tienda_integrada.loc[fechas_comparables_tienda, 'fecha_actualizacion']
    > tienda_integrada.loc[fechas_comparables_tienda, 'fecha_pedido']
)

# Mismo criterio que en el lote: la antiguedad se calcula despues de unir el CRM,
# y la llave del CRM no se entrega duplicada.
tienda_integrada['antiguedad_cliente_dias'] = (
    tienda_integrada['fecha_pedido'] - tienda_integrada['fecha_inscripcion_club']
).dt.days
tienda_integrada['antiguedad_cliente_dias'] = tienda_integrada['antiguedad_cliente_dias'].where(
    tienda_integrada['antiguedad_cliente_dias'].ge(0)
)
tienda_integrada = tienda_integrada.drop(columns='cliente_id')
print(
    f"antiguedad_cliente_dias en la tienda: {int(tienda_integrada['antiguedad_cliente_dias'].notna().sum()):,} "
    f'de {len(tienda_integrada):,} pedidos.'
)

faltantes_esquema_tienda = set(lote_integrado.columns) - set(tienda_integrada.columns)
extras_esquema_tienda = set(tienda_integrada.columns) - set(lote_integrado.columns)
if faltantes_esquema_tienda or extras_esquema_tienda:
    raise ValueError(
        f'Esquema incompatible: faltan {sorted(faltantes_esquema_tienda)}, '
        f'sobran {sorted(extras_esquema_tienda)}.'
    )
tienda_integrada = tienda_integrada.reindex(columns=lote_integrado.columns)
for columna in lote_integrado.columns:
    dtype_esperado = lote_integrado[columna].dtype
    if tienda_integrada[columna].dtype != dtype_esperado:
        antes = tienda_integrada[columna].copy()
        tienda_integrada[columna] = tienda_integrada[columna].astype(dtype_esperado)
        nuevos_vacios = int((antes.notna() & tienda_integrada[columna].isna()).sum())
        print(
            f'Formato {columna}: {antes.dtype} -> {dtype_esperado}; '
            f'{nuevos_vacios:,} valores no vacíos se volvieron vacíos.'
        )
        if nuevos_vacios:
            raise ValueError(f'La alineación de {columna} perdió {nuevos_vacios:,} valores.')

assert list(tienda_integrada.columns) == list(lote_integrado.columns)
assert tienda_integrada.dtypes.equals(lote_integrado.dtypes)
assert tienda_integrada['id_pedido'].is_unique
assert not tienda_integrada.duplicated().any()
assert 'correo_cliente' not in tienda_integrada.columns
assert 'email_cliente' not in tienda_integrada.columns
tienda_con_lote = pd.concat([lote_integrado, tienda_integrada], ignore_index=True)
assert len(tienda_con_lote) == len(lote_integrado) + len(tienda_integrada)
assert tienda_con_lote['id_pedido'].is_unique

tienda_duplicados = int(tienda_cruda.duplicated().sum())
tienda_filas_imposibles = max(0, len(tienda_cruda) - tienda_duplicados - len(tienda_limpia))
tienda_etiquetas_vacias = int(tienda_integrada['fuga_cliente'].isna().sum())
tienda_coincidencias_crm = int(tienda_integrada['crm_encontrada'].sum())
tienda_crm_sin_pareja = int((~tienda_integrada['crm_encontrada']).sum())
tienda_fechas_comparables = (
    tienda_integrada['crm_encontrada']
    & tienda_integrada['fecha_pedido'].notna()
    & tienda_integrada['fecha_actualizacion'].notna()
)
tienda_actualizaciones_posteriores = int(
    tienda_integrada['crm_actualizacion_posterior_pedido'].fillna(False).sum()
)
tienda_bodega_sin_pareja = int((~tienda_integrada['bodega_encontrada']).sum())
tienda_montos_capados = int(tienda_integrada['monto_compra_capado'].sum())
tienda_ventas_corporativas = int(tienda_integrada['es_venta_corporativa'].sum())
tienda_fechas_invalidas = int(tienda_integrada['fecha_pedido'].isna().sum())
tienda_edad_imposible = int(tienda_integrada['edad_cliente_imposible'].sum())
tienda_peso_imposible = int(tienda_integrada['peso_pedido_kg_imposible'].sum())
tienda_ciudad_popayan = int(tienda_integrada['ciudad_tienda'].eq('Popayan').sum())
print(
    f'Tienda nueva lista: {len(tienda_integrada):,} filas, {tienda_integrada.shape[1]} columnas; '
    f'{tienda_duplicados:,} duplicados exactos retirados; '
    f'{tienda_con_lote["id_pedido"].nunique():,} pedidos únicos al anexar.'
)
print(
    f'Sin etiqueta fuga: {tienda_etiquetas_vacias:,}; CRM: {tienda_coincidencias_crm:,} con pareja y '
    f'{tienda_crm_sin_pareja:,} sin pareja; bodega sin pareja: {tienda_bodega_sin_pareja:,}.'
)
print(
    f'Actualización CRM posterior al pedido: {tienda_actualizaciones_posteriores:,} de '
    f'{int(tienda_fechas_comparables.sum()):,} con ambas fechas observadas.'
)
print(f'Ciudad Popayan normalizada: {tienda_ciudad_popayan:,} pedidos; sin fechas inválidas: {tienda_fechas_invalidas:,}.')
verificar_resultado(
    tienda_integrada, 'Tienda nueva integrada',
    vacias_esperadas=('fuga_cliente', 'antiguedad_cliente_dias'),
)
auditoria_tienda = auditar_lote_vs_semestre(
    tienda_integrada, referencia=referencia, titulo='tienda nueva Popayan'
)
ruta_tienda_salida = S17 / 'S17_Sudo_Sancocho_Lote08_tienda_nueva.csv'
tienda_integrada.to_csv(ruta_tienda_salida, index=False, encoding='utf-8-sig')
print(f'Entregable escrito: {ruta_tienda_salida.name} ({len(tienda_integrada):,} filas)')

Archivo de tienda nueva: 3,060 filas y 19 columnas
Esquema de columnas normalizado: 19 nombres; alias canal -> canal_compra aplicado=True.
Lote crudo: 3,060 filas, 19 columnas.
Duplicados exactos: 60; filas tras eliminarlos: 3,000.
Conversion id_pedido: 0 valores no vacios pasaron a vacios.
Conversion id_cliente: 0 valores no vacios pasaron a vacios.
Conversion codigo_postal: 0 valores no vacios pasaron a vacios.
Conversion precio_unitario: 0 valores no vacios pasaron a vacios.
Conversion unidades_vendidas: 0 valores no vacios pasaron a vacios.
Conversion edad_cliente: 0 valores no vacios pasaron a vacios.
Conversion peso_pedido_kg: 0 valores no vacios pasaron a vacios.
Conversion monto_compra: 0 valores no vacios pasaron a vacios.
Conversion fecha_pedido: 0 valores no vacios pasaron a vacios.
Conversion fecha_actualizacion_stock: 0 valores no vacios pasaron a vacios.


Normalizacion canal_compra: 0 vacios nuevos; 4 categorias.


Normalizacion metodo_pago: 0 vacios nuevos; 4 categorias.


Normalizacion ciudad_tienda: 0 vacios nuevos; 1 categorias.


Normalizacion categoria_producto: 0 vacios nuevos; 6 categorias.
Normalizacion nivel_satisfaccion: 0 vacios nuevos; 3 categorias.


Normalizacion nivel_lealtad: 0 vacios nuevos; 4 categorias.
Valores imposibles de pedido: precio negativo/centinela=0, unidades no positivas/centinela=0, monto negativo=0; filas eliminadas (interseccion incluida)=0.
Regla monto_compra = precio_unitario x unidades_vendidas: 0 valores discrepantes; se recalculan.
Edad fuera de 0-100: 0; peso mayor de 100 kg: 0; se conservan los pedidos y se dejan esos valores vacios.
Precio imputado con mediana semestral (305,600.00): 0; monto recalculado/imputado: 0.
Satisfaccion imputada por moda semestral del canal: 0 (incluye canal nuevo con moda global si aplica).
Correo completado con marcador explicito: 0; comentarios completados: 0; ciudad sin imputar (con indicador): 0.
Outliers segun IQR semestral: corporativas marcadas=147; montos capados con limite 1,669,600.00=135.


... (2 outputs left out)

Vacíos:


,cantidad
fuga_cliente,3000
puntos_fidelidad,2580
fecha_actualizacion,2580
fecha_inscripcion_club,2580
crm_actualizacion_posterior_pedido,2580
segmento_crm,2580
antiguedad_cliente_dias,2580
ticket_promedio_previo_cliente,1289
dias_desde_compra_previa,1289
proveedor_principal,503


canal_compra: ['App', 'Marketplace', 'Tienda', 'Web']
metodo_pago: ['Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']
categoria_producto: ['Belleza', 'Deportes', 'Electronica', 'Hogar', 'Jugueteria', 'Moda']
ciudad_tienda: ['Popayan']
nivel_satisfaccion: ['Alto', 'Bajo', 'Medio']
nivel_lealtad: ['Bronce', 'Oro', 'Plata', 'Platino']
Columnas 100% vacias: ['fuga_cliente']

Auditoria tienda nueva Popayan frente al semestre (3,000 filas):
Columnas solo en el lote: ['crm_actualizacion_posterior_pedido', 'fecha_actualizacion', 'puntos_fidelidad']
Columnas solo en el semestre: ['acepta_promociones', 'ciudad_residencia', 'puntos_fidelidad_actuales']
Diferencias de tipos (lote, semestre): {'bodega_asignada': ('str', 'object'), 'categoria': ('str', 'object'), 'fecha_actualizacion_stock': ('datetime64[us]', 'datetime64[ns]'), 'fecha_inscripcion_club': ('datetime64[us]', 'datetime64[ns]'), 'fecha_pedido': ('datetime64[us]', 'datetime64[ns]'), 'proveedor_principal': ('str', 'object'), 'segmento_crm

Diferencias de categorias: {'ciudad_tienda': {'nuevas_en_lote': ['Popayan'], 'no_observadas_en_lote': ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']}}
Diferencias de medianas (lote vs. semestre): {'precio_unitario': {'mediana_lote': 373350.0, 'mediana_semestre': 305600.0, 'diferencia': 67750.0}, 'unidades_vendidas': {'mediana_lote': 2.0, 'mediana_semestre': 2.0, 'diferencia': 0.0}, 'monto_compra': {'mediana_lote': 565100.0, 'mediana_semestre': 476400.0, 'diferencia': 88700.0}, 'edad_cliente': {'mediana_lote': 46.0, 'mediana_semestre': 46.0, 'diferencia': 0.0}, 'peso_pedido_kg': {'mediana_lote': 11.5, 'mediana_semestre': 11.5, 'diferencia': 0.0}}
Diferencias de porcentaje de vacios (puntos porcentuales): {'id_pedido': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'id_cliente': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'fe

## 6. Acta de incidentes

La celda siguiente prioriza hallazgos de impacto y registra las cinco partes por incidente. Cada evidencia se calcula desde los lotes y las solicitudes de los minutos 50 y 80.

In [8]:
n = len(lote_crudo)
duplicados = int(lote_crudo.duplicated().sum())
# filas_eliminadas_por_imposibles viene del conteo real de limpiar_lote (seccion 3).
edad_imposible = int(lote_limpio['edad_cliente_imposible'].sum())
peso_imposible = int(lote_limpio['peso_pedido_kg_imposible'].sum())
crm_sin_pareja = sin_pareja_crm
bodega_sin_pareja = int((~lote_limpio['bodega_encontrada']).sum())
montos_capados = int(lote_limpio['monto_compra_capado'].sum())
ventas_corporativas = int(lote_limpio['es_venta_corporativa'].sum())
clientes_lote_total = lote_limpio['id_cliente'].nunique()
clientes_crm_compartidos = lote_integrado.loc[lote_integrado['crm_encontrada'], 'id_cliente'].nunique()
clientes_semestre_total = len(clientes_semestre)
fechas_pedido_invalidas = int(lote_limpio['fecha_pedido'].isna().sum())
etiquetas_fuga_vacias = int(lote_limpio['fuga_cliente'].isna().sum()) if 'fuga_cliente' in lote_limpio else 0
tienda_etiquetas_vacias = int(tienda_integrada['fuga_cliente'].isna().sum())
tienda_sin_etiqueta_fuente = 'fuga_cliente' not in tienda_cruda.columns
tienda_crm_sin_pareja = int((~tienda_integrada['crm_encontrada']).sum())
tienda_bodega_sin_pareja = int((~tienda_integrada['bodega_encontrada']).sum())
tienda_bodega_sin_pareja_por_categoria = tienda_integrada.loc[
    ~tienda_integrada['bodega_encontrada'], 'categoria_producto'
].value_counts().to_dict()
tienda_montos_capados = int(tienda_integrada['monto_compra_capado'].sum())
tienda_ventas_corporativas = int(tienda_integrada['es_venta_corporativa'].sum())
tienda_duplicados = int(tienda_cruda.duplicated().sum())
tienda_filas_imposibles = max(0, len(tienda_cruda) - tienda_duplicados - len(tienda_limpia))
tienda_fechas_invalidas = int(tienda_integrada['fecha_pedido'].isna().sum())
tienda_edad_imposible = int(tienda_integrada['edad_cliente_imposible'].sum())
tienda_peso_imposible = int(tienda_integrada['peso_pedido_kg_imposible'].sum())
nuevas_categorias = {}
for columna, categorias_semestre in referencia['semestre']['categorias'].items():
    categorias_observadas = set(lote_limpio[columna].dropna().astype(str)) | set(tienda_integrada[columna].dropna().astype(str))
    nuevas = categorias_observadas - set(categorias_semestre)
    for categoria in sorted(nuevas):
        cantidad = int(lote_limpio[columna].eq(categoria).sum() + tienda_integrada[columna].eq(categoria).sum())
        nuevas_categorias[f'{columna}={categoria}'] = cantidad

incidentes = [
    {
        'impacto': 2,
        'qué encontraste': 'Pedidos fechados fuera del periodo que declara el lote',
        'evidencia': f'Solo {pedidos_en_periodo:,} de {n:,} pedidos ({pedidos_en_periodo / max(n, 1):.1%}) caen en {periodo_declarado}; {pedidos_antes_periodo:,} son anteriores y {pedidos_despues_periodo:,} posteriores, de los cuales {pedidos_anio_siguiente:,} quedan en {periodo_declarado.year + 1} o despues.',
        'decisión': 'Conservar los pedidos, marcar el hallazgo y no usar el lote como corte mensual de noviembre sin confirmarlo con la fuente.',
        'por qué': 'Una fecha posterior a la entrega del lote no puede corresponder a una venta ya ocurrida, y el grueso del archivo es de octubre, no de noviembre.',
        'riesgo que queda': 'Recencia, dias_desde_compra_previa y rfm_score se calculan con esas fechas; cualquier informe mensual o de tendencia queda distorsionado hasta aclarar el origen.'
    },
    {
        'impacto': 7,
        'qué encontraste': f'Vacios de nivel_satisfaccion concentrados en el canal {satisfaccion_peor_canal}',
        'evidencia': f'{satisfaccion_vacios_pct:.2f}% de vacios en el lote frente a {satisfaccion_semestre_pct:.2f}% del semestre, casi igual en global; pero {satisfaccion_peor_canal} tiene {satisfaccion_peor_canal_pct:.1f}% y concentra el {satisfaccion_share_peor_canal:.1f}% de todos los vacios, como en el semestre, donde {satisfaccion_peor_canal_semestre} concentraba el {satisfaccion_share_semestre_peor:.1f}%.',
        'decisión': 'Mantener el tratamiento MAR de S04, que imputa por moda segun canal, en vez de una moda global.',
        'por qué': 'El vacio depende del canal y no del nivel de satisfaccion en si; el mecanismo del semestre sigue vigente en este lote.',
        'riesgo que queda': f'Si la encuesta de {satisfaccion_peor_canal} sigue fallando, la satisfaccion imputada de ese canal se parecera siempre a su moda y subestimara su variabilidad real.'
    },
    {
        'impacto': 1,
        'qué encontraste': 'Montos altos modificados por el límite IQR',
        'evidencia': f'Lote base: {montos_capados:,} montos capados, {ventas_corporativas:,} corporativas; tienda nueva: {tienda_montos_capados:,} capados y {tienda_ventas_corporativas:,} corporativas; límite semestral {referencia["limite_superior_iqr_monto"]:,.0f} pesos.',
        'decisión': 'Aplicar la regla S07: capar 2-3 unidades y conservar/etiquetar 4-5 unidades.',
        'por qué': 'Se mantiene la política documentada del semestre y no se descartan ventas de alto valor.',
        'riesgo que queda': 'El cap modifica ingresos; conciliar esos pedidos con el CSV crudo antes de un informe financiero.'
    },
    {
        'impacto': 9,
        'qué encontraste': 'Etiqueta fuga_cliente incompleta',
        'evidencia': f'Lote base: {etiquetas_fuga_vacias:,} de {len(lote_limpio):,} sin etiqueta; tienda nueva: {tienda_etiquetas_vacias:,} de {len(tienda_integrada):,} sin etiqueta y el archivo fuente no incluía la columna ({tienda_sin_etiqueta_fuente}).',
        'decisión': 'Conservar los valores conocidos como objetivo y no imputar los faltantes.',
        'por qué': 'La etiqueta representa un resultado y no debe inferirse a partir de predictores.',
        'riesgo que queda': 'No entrenar ni evaluar un modelo supervisado de fuga hasta aclarar la cobertura.'
    },
    {
        'impacto': 3,
        'qué encontraste': 'Parte de los datos CRM se actualizó después del pedido',
        'evidencia': f'Lote base: {actualizaciones_posteriores:,} de {int(fechas_comparables.sum()):,}; tienda nueva: {tienda_actualizaciones_posteriores:,} de {int(tienda_fechas_comparables.sum()):,} emparejados tienen actualización posterior al pedido.',
        'decisión': 'Conservar la fecha y la marca temporal; no usar esos campos CRM como predictores históricos en esas filas.',
        'por qué': 'La foto CRM puede contener información conocida después de la compra y filtrar el futuro al modelar.',
        'riesgo que queda': 'La fecha de actualización no revela cuándo cambió cada atributo; se necesita historial CRM con vigencia por campo.'
    },
    {
        'impacto': 4,
        'qué encontraste': 'Llaves duplicadas y un empate conflictivo en el CRM',
        'evidencia': f'{llaves_repetidas:,} llaves aparecen repetidas en {filas_repetidas:,} filas; {len(llaves_ambiguas):,} llave(s) siguen en conflicto y afectan {pedidos_llave_ambigua:,} pedidos.',
        'decisión': 'Usar la fecha más reciente para llaves resolubles y excluir la llave empatada conflictiva.',
        'por qué': f'No hay evidencia para escoger una de las dos versiones de {sorted(llaves_ambiguas)}.',
        'riesgo que queda': 'La fecha debe ser confiable; clientes con empates futuros seguirán sin datos hasta que CRM los concilie.'
    },
    {
        'impacto': 5,
        'qué encontraste': 'Pedidos con valores de venta imposibles',
        'evidencia': f'Lote base: {filas_eliminadas_por_imposibles:,} filas imposibles y {duplicados:,} duplicados exactos; tienda nueva: {tienda_filas_imposibles:,} imposibles y {tienda_duplicados:,} duplicados exactos retirados; reglas S03/S07.',
        'decisión': 'Excluir filas con precio negativo/centinela, unidades no positivas/centinela o monto negativo.',
        'por qué': 'No representan una venta interpretable según las reglas de calidad ya documentadas.',
        'riesgo que queda': 'Un pedido real mal capturado podría haber sido excluido; revisar con la fuente antes de reportar ingresos.'
    },
    {
        'impacto': 6,
        'qué encontraste': 'Categorías que no existían en el semestre',
        'evidencia': f'{len(nuevas_categorias)} valores nuevos; conteos por valor: {nuevas_categorias}.',
        'decisión': 'Conservar cada valor nuevo en forma canónica; no mapearlo a otra categoría.',
        'por qué': 'Son valores válidos de noviembre o de la tienda nueva; la ubicación Popayan no debe mapearse a otra ciudad.',
        'riesgo que queda': 'Los reportes históricos y reglas de imputación pueden no conocer estas categorías.'
    },
    {
        'impacto': 8,
        'qué encontraste': 'Pedidos sin pareja en CRM',
        'evidencia': f'Lote integrado: {crm_sin_pareja:,} de {len(lote_integrado):,} sin pareja; tienda nueva: {tienda_coincidencias_crm:,} con pareja y {tienda_crm_sin_pareja:,} de {len(tienda_integrada):,} sin pareja.',
        'decisión': 'Conservar pedidos mediante left join y dejar las variables CRM vacías.',
        'por qué': 'La unión validada m:1 conserva una fila por pedido sin inventar datos del cliente.',
        'riesgo que queda': 'Las features de cliente no son comparables para filas sin CRM.'
    },
    {
        'impacto': 10,
        'qué encontraste': 'Sin clientes compartidos con el semestre',
        'evidencia': f'{clientes_compartidos:,} de {clientes_lote_total:,} clientes aparecen entre {clientes_semestre_total:,} históricos; el CRM nuevo sí empareja {clientes_crm_compartidos:,} clientes del lote.',
        'decisión': 'Interpretar las features históricas como historial dentro del lote solamente.',
        'por qué': 'Los ids del lote no tienen historial en el semestre; el CRM nuevo es una fuente distinta y no reemplaza el historial transaccional.',
        'riesgo que queda': 'RFM, frecuencia y gasto previo no representan el historial completo del cliente.'
    },
    {
        'impacto': 11,
        'qué encontraste': 'Fechas de pedido inválidas',
        'evidencia': f'Lote base: {fechas_pedido_invalidas:,} fechas inválidas; tienda nueva: {tienda_fechas_invalidas:,} de {len(tienda_integrada):,} fechas inválidas.',
        'decisión': 'Conservar pedidos y dejar la fecha como NaT; no adivinar una fecha.',
        'por qué': 'Sin fecha no se pueden ordenar pedidos ni calcular features temporales con evidencia.',
        'riesgo que queda': 'Excluir estas filas de análisis temporal, recencia e historial.'
    },
    {
        'impacto': 12,
        'qué encontraste': 'Categorías sin pareja en catálogo de bodega',
        'evidencia': f'Lote base: {bodega_sin_pareja:,} de {len(lote_limpio):,} sin bodega; tienda nueva: {tienda_bodega_sin_pareja:,} de {len(tienda_integrada):,}, por categoría: {tienda_bodega_sin_pareja_por_categoria}.',
        'decisión': 'Conservar el pedido y no asignar bodega/proveedor sin evidencia.',
        'por qué': 'El catálogo no cubre todas las categorías del lote.',
        'riesgo que queda': 'Planificación de inventario incompleta para las categorías sin pareja.'
    },
    {
        'impacto': 13,
        'qué encontraste': 'Edad o peso fuera del rango físico revisado',
        'evidencia': f'Lote base: {edad_imposible:,} edades y {peso_imposible:,} pesos imposibles; tienda nueva: {tienda_edad_imposible:,} edades y {tienda_peso_imposible:,} pesos imposibles.',
        'decisión': 'Convertir solo esos valores a faltante y mantener indicadores de origen.',
        'por qué': 'El pedido sigue siendo útil para ventas; edad y peso no se corrigen por conjetura.',
        'riesgo que queda': 'Los análisis demográficos/logísticos deben excluir o tratar explícitamente esos faltantes.'
    },
    {
        'impacto': 14,
        'qué encontraste': 'Duplicados exactos',
        'evidencia': f'Lote base: {duplicados:,} de {n:,} duplicados; tienda nueva: {tienda_duplicados:,} de {len(tienda_cruda):,}; quedan {len(lote_limpio) + len(tienda_integrada):,} pedidos únicos al anexar.',
        'decisión': 'Conservar una copia idéntica por pedido; no eliminar compras por correo repetido.',
        'por qué': 'Las copias exactas duplican un evento; correos repetidos pueden ser compras recurrentes.',
        'riesgo que queda': 'Un duplicado no exacto con id_pedido distinto requiere validación externa.'
    },
    {
        'impacto': 15,
        'qué encontraste': 'Valores de ciudad no registrados',
        'evidencia': f'{int(lote_limpio["ciudad_tienda"].isna().sum()):,} vacíos ({lote_limpio["ciudad_tienda"].isna().mean():.2%}) entre {len(lote_limpio):,} pedidos; se compara con el semestre en la tabla de vacíos.',
        'decisión': 'Dejar ciudad vacía y conservar ciudad_tienda_imputado.',
        'por qué': 'S04 dejó esta variable geográfica sin imputar para no inventar residencia/origen.',
        'riesgo que queda': 'Los análisis geográficos representan solo los pedidos con ciudad observada.'
    }
]
# 'impacto' solo ordena el acta de mayor a menor gravedad; no se muestra.
acta = pd.DataFrame(incidentes).sort_values('impacto').drop(columns='impacto')
display(acta)

print(
    f'Recomendación: lote con {len(lote_limpio):,} pedidos únicos de {n:,} filas crudas. '
    f'Puede usarse solo para análisis descriptivo de ventas después de documentar las '
    f'{filas_eliminadas_por_imposibles:,} exclusiones por valores imposibles, '
    f'conciliar los {montos_capados:,} montos capados con el archivo crudo para cualquier total financiero, '
    f'Tienda nueva: {len(tienda_integrada):,} pedidos anexables, {tienda_montos_capados:,} montos capados, '
    f'{tienda_etiquetas_vacias:,} etiquetas ausentes, {tienda_coincidencias_crm:,} con CRM, '
    f'{tienda_crm_sin_pareja:,} sin CRM y {tienda_actualizaciones_posteriores:,} actualizaciones posteriores; '
    f'{tienda_bodega_sin_pareja:,} sin bodega; usarla solo para análisis descriptivo hasta completar esas fuentes. '
    f'validar las categorías nuevas y tratar por separado las filas sin CRM/bodega. '
    f'CRM nuevo: {coincidencias:,} pedidos emparejados, {crm_sin_pareja:,} sin pareja y {actualizaciones_posteriores:,} actualizaciones posteriores al pedido; '
    f'no entrenar un modelo con {etiquetas_fuga_vacias:,} etiquetas faltantes ni usar variables posteriores al pedido como predictores; '
    f'además, solo {clientes_compartidos:,} clientes del lote tienen id compartido con el semestre; '
    'antes del próximo lote, completar etiquetas, actualizar catálogos y revisar la cobertura de llaves.'
)

print()
print('Bitacora de IA')
print('Herramientas y para que se usaron:')
print('  - Copilot SDK en VS Code: contrastar las reglas de S03 a S13 y preparar funciones_novamarket.py y el notebook.')
print('  - Claude, en Claude Code: auditar el notebook y los tres entregables contra la rubrica, y aplicar las correcciones que salieron de esa auditoria.')
print('  - Python y pandas: ejecutar conversiones, conteos y comprobaciones. Cada numero de esta acta sale de una celda ejecutada, no de la herramienta.')
print()
print('Que salio de la auditoria con IA y se aplico:')
print(f'  - Dimension de oportunidad, que no estabamos midiendo: {pedidos_anio_siguiente:,} pedidos fechados en {periodo_declarado.year + 1} o despues, y solo {pedidos_en_periodo:,} dentro del periodo que declara el nombre del archivo. Es el incidente de impacto 2.')
print(f'  - Vacios de nivel_satisfaccion por canal: {satisfaccion_peor_canal} concentra el {satisfaccion_share_peor_canal:.1f}% de los vacios, frente al {satisfaccion_share_semestre_peor:.1f}% del semestre. Confirma que el mecanismo MAR de S04 sigue vigente.')
print('  - Assert de columnas 100% vacias en verificar_resultado. Destapo 6 columnas del CRM del semestre que ibamos a entregar vacias en el preprocesado.')
print('  - Recalcular antiguedad_cliente_dias despues de unir el CRM del lote, que antes salia vacia en los tres archivos.')
print()
print('Propuestas de la IA que NO se siguieron:')
print('  1. Imputar ciudad_tienda con la moda. Se comprobo que S04 deja la ciudad sin imputar y crea ciudad_tienda_imputado; se conserva ese criterio para no asignar una geografia sin evidencia.')
print(f'  2. Agregar Popayan al diccionario de ciudades, porque la IA afirmo que si no quedaria vacia. Se leyo _normalizar_ciudad: devuelve reemplazos.get(texto, texto).title(), asi que una ciudad desconocida pasa en forma canonica. Se comprobo sobre el archivo y los {len(tienda_integrada):,} pedidos quedaron en Popayan sin tocar el diccionario. La afirmacion era falsa y el cambio habria sido innecesario.')
print('  3. Usar una funcion aparte, adaptar_esquema_externo, para renombrar las columnas de la tienda. Se descarto frente a _normalizar_nombres_columnas dentro de limpiar_lote, porque el kit exige que limpiar_lote reciba el CSV crudo sin pasos previos que alguien pueda olvidar.')
print()
print('Criterio del equipo: cada sugerencia de la IA se comprobo contra el lote antes de aplicarla, y las que no resistieron la comprobacion se descartaron, como el caso 2.')


,qué encontraste,evidencia,decisión,por qué,riesgo que queda
2,Montos altos modificados por el límite IQR,"Lote base: 844 montos capados, 995 corporativa...",Aplicar la regla S07: capar 2-3 unidades y con...,Se mantiene la política documentada del semest...,El cap modifica ingresos; conciliar esos pedid...
0,Pedidos fechados fuera del periodo que declara...,"Solo 1,736 de 20,300 pedidos (8.6%) caen en 20...","Conservar los pedidos, marcar el hallazgo y no...",Una fecha posterior a la entrega del lote no p...,"Recencia, dias_desde_compra_previa y rfm_score..."
4,Parte de los datos CRM se actualizó después de...,"Lote base: 555 de 15,400; tienda nueva: 12 de ...",Conservar la fecha y la marca temporal; no usa...,La foto CRM puede contener información conocid...,La fecha de actualización no revela cuándo cam...
5,Llaves duplicadas y un empate conflictivo en e...,"509 llaves aparecen repetidas en 1,018 filas; ...",Usar la fecha más reciente para llaves resolub...,No hay evidencia para escoger una de las dos v...,La fecha debe ser confiable; clientes con empa...
6,Pedidos con valores de venta imposibles,"Lote base: 2,196 filas imposibles y 300 duplic...","Excluir filas con precio negativo/centinela, u...",No representan una venta interpretable según l...,Un pedido real mal capturado podría haber sido...
7,Categorías que no existían en el semestre,3 valores nuevos; conteos por valor: {'canal_c...,Conservar cada valor nuevo en forma canónica; ...,Son valores válidos de noviembre o de la tiend...,Los reportes históricos y reglas de imputación...
1,Vacios de nivel_satisfaccion concentrados en e...,7.25% de vacios en el lote frente a 7.26% del ...,"Mantener el tratamiento MAR de S04, que imputa...",El vacio depende del canal y no del nivel de s...,"Si la encuesta de App sigue fallando, la satis..."
8,Pedidos sin pareja en CRM,"Lote integrado: 1,849 de 17,804 sin pareja; ti...",Conservar pedidos mediante left join y dejar l...,La unión validada m:1 conserva una fila por pe...,Las features de cliente no son comparables par...
3,Etiqueta fuga_cliente incompleta,"Lote base: 14,249 de 17,804 sin etiqueta; tien...",Conservar los valores conocidos como objetivo ...,La etiqueta representa un resultado y no debe ...,No entrenar ni evaluar un modelo supervisado d...
9,Sin clientes compartidos con el semestre,"0 de 4,897 clientes aparecen entre 152,176 his...",Interpretar las features históricas como histo...,Los ids del lote no tienen historial en el sem...,"RFM, frecuencia y gasto previo no representan ..."


Recomendación: lote con 17,804 pedidos únicos de 20,300 filas crudas. Puede usarse solo para análisis descriptivo de ventas después de documentar las 2,196 exclusiones por valores imposibles, conciliar los 844 montos capados con el archivo crudo para cualquier total financiero, Tienda nueva: 3,000 pedidos anexables, 135 montos capados, 3,000 etiquetas ausentes, 420 con CRM, 2,580 sin CRM y 12 actualizaciones posteriores; 503 sin bodega; usarla solo para análisis descriptivo hasta completar esas fuentes. validar las categorías nuevas y tratar por separado las filas sin CRM/bodega. CRM nuevo: 15,955 pedidos emparejados, 1,849 sin pareja y 555 actualizaciones posteriores al pedido; no entrenar un modelo con 14,249 etiquetas faltantes ni usar variables posteriores al pedido como predictores; además, solo 0 clientes del lote tienen id compartido con el semestre; antes del próximo lote, completar etiquetas, actualizar catálogos y revisar la cobertura de llaves.

Bitacora de IA
Herramientas y